In [14]:
import pandas as pd
import numpy as np

In [2]:
log_returns = pd.read_csv("data/log_returns.csv", index_col=0, parse_dates=True)
target = pd.read_csv("data/target.csv", index_col=0, parse_dates=True)

print(log_returns.shape)
print(target.shape)

(2764, 462)
(2739, 462)


In [7]:
subset = [
    "AAPL", "MSFT", "NVDA", "JPM", "JNJ",
    "XOM", "HD", "GOOGL", "BAC", "PFE"
]

returns_subset = log_returns[subset]
target_subset = target[subset]

print(f"subset size: {len(subset)} stocks")

subset size: 10 stocks


In [8]:
train_end = "2022-12-31"
test_start = "2023-01-01"

train_returns = returns_subset.loc[:train_end]
test_returns  = returns_subset.loc[test_start:]
test_target   = target_subset.loc[test_start:]

print(f"train: {train_returns.shape}")
print(f"test:  {test_returns.shape}")

train: (2013, 10)
test:  (751, 10)


In [10]:
from pmdarima import auto_arima
import warnings
warnings.filterwarnings("ignore")

arima_models = {}

for i, ticker in enumerate(subset):
    model = auto_arima(
        train_returns[ticker],
        seasonal=False,
        stepwise=True,
        max_p=3, max_q=3,
        suppress_warnings=True,
        error_action="ignore"
    )
    arima_models[ticker] = model

    print(f"fitted {ticker} - {i + 1}/{len(subset)}")

print("done")

fitted AAPL - 1/10
fitted MSFT - 2/10
fitted NVDA - 3/10
fitted JPM - 4/10
fitted JNJ - 5/10
fitted XOM - 6/10
fitted HD - 7/10
fitted GOOGL - 8/10
fitted BAC - 9/10
fitted PFE - 10/10
done


In [12]:
predictions = {}

for ticker in subset:
    model = arima_models[ticker]
    forecast = model.predict(n_periods=len(test_returns))
    predictions[ticker] = forecast.values if hasattr(forecast, 'values') else forecast

predictions_df = pd.DataFrame(predictions, index=test_returns.index)
print(predictions_df.shape)
print(predictions_df.head())

(751, 10)
                AAPL      MSFT      NVDA       JPM       JNJ  XOM        HD  \
Date                                                                          
2023-01-03 -0.001835  0.001181  0.001763 -0.000305  0.000902  0.0  0.002426   
2023-01-04  0.000872  0.000876  0.001686  0.000434  0.000317  0.0 -0.000025   
2023-01-05 -0.000958  0.000876  0.001691 -0.000229  0.000374  0.0  0.000168   
2023-01-06  0.000789  0.000876  0.001691  0.000000  0.000369  0.0  0.001104   
2023-01-09 -0.000705  0.000876  0.001691  0.000000  0.000369  0.0  0.000194   

               GOOGL       BAC           PFE  
Date                                          
2023-01-03  0.000682  0.002196  7.370080e-05  
2023-01-04  0.000593 -0.001512 -3.094748e-06  
2023-01-05  0.000593  0.000904  1.299506e-07  
2023-01-06  0.000593 -0.000309 -5.456715e-09  
2023-01-09  0.000593 -0.000174  2.291312e-10  


In [17]:
from sklearn.metrics import mean_squared_error

mse_scores = {}
directional_accuracy = {}

for ticker in subset:
    actual = test_target[ticker].dropna()
    pred = predictions_df[ticker].loc[actual.index]

    mse_scores[ticker] = mean_squared_error(actual, pred)
    directional_accuracy[ticker] = (np.sign(actual) == np.sign(pred)).mean()

results_arima = pd.DataFrame({
    "MSE": mse_scores,
    "RMSE": {t: np.sqrt(v) for t, v in mse_scores.items()},
    "Directional Accuracy": directional_accuracy
})

print(results_arima)
print(f"\nmean MSE:  {results_arima['MSE'].mean():.6f}")
print(f"mean RMSE: {results_arima['RMSE'].mean():.6f}")
print(f"mean directional accuracy: {results_arima['Directional Accuracy'].mean():.4f}")

            MSE      RMSE  Directional Accuracy
AAPL   0.001442  0.037973              0.505362
MSFT   0.000945  0.030741              0.607239
NVDA   0.004503  0.067107              0.620643
JPM    0.001038  0.032213              0.001340
JNJ    0.000560  0.023664              0.509383
XOM    0.001019  0.031925              0.000000
HD     0.000954  0.030887              0.529491
GOOGL  0.001774  0.042119              0.589812
BAC    0.001484  0.038522              0.500000
PFE    0.001138  0.033729              0.155496

mean MSE:  0.001486
mean RMSE: 0.036888
mean directional accuracy: 0.4019


In [18]:
results_arima.to_csv("data/arima_results.csv")
print("saved")

saved
